# testing

## testing

In [ ]:
open sm'_operators

### testing_trace

In [ ]:
union testing_trace =
    | Console
    | Trace
    | TraceRaw
    | Silent

### __expect

In [ ]:
inl rec __expect
    (fn : _ -> _ -> bool)
    (trace' : testing_trace)
    (name : string)
    (b : _)
    (a : _)
    : ()
    =
    inl result = fn a b
    inl get_data () =
        if result
        then sm'.show { expected = b }
        else sm'.show { actual = a; expected = b }
    inl get_raw_text () =
        inl data = get_data ()
        name ++# " " ++# get_data ()
    inl result =
        result || join result
    match trace' with
    | Console =>
        inl text = get_raw_text ()
        text |> console.write_line
        text
    | Trace =>
        trace Info (fun () => name) get_data
        get_raw_text ()
    | TraceRaw =>
        inl text = get_raw_text ()
        trace_raw Info fun () => text
        text
    | Silent => reflection.nameof { __expect }
    |> assert result

### __assert_approx_eq

In [ ]:
inl rec __assert_approx_eq trace e b a =
    __expect
        (fun a b => abs (b - a) < (e |> optionm.defaultWith 0.00000001))
        trace
        (reflection.nameof { __assert_approx_eq })
        b
        a

inl _assert_approx_eq e b a =
    __assert_approx_eq Console e b a

In [1]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

12.345f64
|> _assert_approx_eq (Some 0.0001f64) 12.345f64

.gleam (gleam)
__assert_approx_eq { expected = 12.345 }
.fsx (fsharp)
__assert_approx_eq { expected = 12.345 }
.py (cuda)
__assert_approx_eq { expected = 12.345 }
.rs (rust)
__assert_approx_eq { expected = 12.345 }
.ts (typescript)
__assert_approx_eq { expected = 12.345 }

0

In [2]:
///- --test --print-code

1f64
|> __assert_approx_eq Console (Some 3) 2

__assert_approx_eq { expected = 2.0 }

In [3]:
///- --test --print-code

(dyn 1f64)
|> _assert_approx_eq (Some 3) 2

__assert_approx_eq { expected = 2.0 }

### __assert_eq

In [ ]:
inl rec __assert_eq trace b a =
    __expect (=) trace (reflection.nameof { __assert_eq }) b a

inl _assert_eq b a =
    __assert_eq Console b a

### __assert_eq'

In [ ]:
inl rec __assert_eq' forall t. (trace : testing_trace) (b : t) (a : t) : () =
    __expect
        (fun a b => #?a = #?b)
        trace
        (reflection.nameof { __assert_eq' = __assert_eq' : _ -> t -> t -> () })
        b
        a

inl _assert_eq' b a =
    __assert_eq' Console b a

### __assert_eq''

In [ ]:
inl rec __assert_eq'' trace b a =
    __expect (=.) trace (reflection.nameof { __assert_eq'' }) b a

inl _assert_eq'' b a =
    __assert_eq'' Console b a

### __assert_ne

In [ ]:
inl rec __assert_ne trace b a =
    __expect (<>.) trace (reflection.nameof { __assert_ne }) b a

inl _assert_ne b a =
    __assert_ne Console b a

### __assert_gt

In [ ]:
inl rec __assert_gt trace b a =
    __expect (>) trace (reflection.nameof { __assert_gt }) b a

inl _assert_gt b a =
    __assert_gt Console b a

### __assert_ge

In [ ]:
inl rec __assert_ge trace b a =
    __expect (>=) trace (reflection.nameof { __assert_ge }) b a

inl _assert_ge b a =
    __assert_ge Console b a

### __assert_lt

In [ ]:
inl rec __assert_lt trace b a =
    __expect (<) trace (reflection.nameof { __assert_lt }) b a

inl _assert_lt b a =
    __assert_lt Console b a

### __assert_le

In [ ]:
inl rec __assert_le trace b a =
    __expect (<=) trace (reflection.nameof { __assert_le }) b a

inl _assert_le b a =
    __assert_le Console b a

### __assert

In [ ]:
inl rec __assert fn trace b a =
    __expect fn trace (reflection.nameof { __assert }) a b

inl _assert fn b a =
    __assert fn Console b a

### __assert_between

In [ ]:
inl rec __assert_between trace a b actual =
    inl assert_between actual (a, b) =
        __assert_ge Silent a actual
        __assert_le Silent b actual
        true
    __expect assert_between trace (reflection.nameof { __assert_between }) (a, b) actual

inl _assert_between a b actual =
    __assert_between Console a b actual

### _assert_fn

In [ ]:
inl rec _assert_fn fn list =
    list
    |> listm.rev
    |> listm.map fun input, expected => join
        input
        |> fn
        |> resultm.get
        |> fun x =>
            inl expected' = join expected
            inl name = reflection.nameof { _assert_fn }
            try
                fun () =>
                    console.write_line ""
                    trace Verbose
                        fun () => name
                        fun () => { input }
                    sm'.show x
                    |> _assert_eq (sm'.show expected')
                    true
                fun ex =>
                    trace Critical
                        fun () => name ++# " / error"
                        fun () => { ex expected }
                    Some false
            |> optionm.value
    |> listm'.filter not
    |> function
        | [] => ()
        | x => #?*x |> failwith

In [4]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

[
    3i32,
    2i32

    2,
    1
]
|> _assert_fn fun x =>
    x - 1
    |> Ok
    |> fun x => x : _ _ ()

.gleam (gleam)
15:44:36 1v _assert_fn / { input = 3 }
__assert_eq { expected = 2 }
15:44:36 1v _assert_fn / { input = 2 }
__assert_eq { expected = 1 }
.fsx (fsharp)
15:44:38 1v _assert_fn / { input = 3 }
__assert_eq { expected = 2 }
15:44:38 2v _assert_fn / { input = 2 }
__assert_eq { expected = 1 }
.py (cuda)
15:44:39 1v _assert_fn / { input = 3 }
__assert_eq { expected = 2 }

15:44:39 2v _assert_fn / { input = 2 }
__assert_eq { expected = 1 }
.rs (rust)

15:44:40 1v _assert_fn / { input = 3 }
__assert_eq { expected = 2 }

15:44:40 2v _assert_fn / { input = 2 }
__assert_eq { expected = 1 }
.ts (typescript)
15:44:41 1v _assert_fn / { input = 3 }
__assert_eq { expected = 2 }
15:44:41 2v _assert_fn / { input = 2 }
__assert_eq { expected = 1 }

0

### _assert_in

In [ ]:
inl _assert_in items x =
    (None, items)
    ||> listm.fold fun acc x' =>
        match acc with
        | Some x => Some x
        | None =>
            if x' <>. x
            then None
            else
                trace Debug (fun () => "_assert_in") fun () => { x x' }
                Some ()
    |> function
        | Some _ => ()
        | None =>
            "_assert_in / " ++#? { x items } |> failwith

## fsharp

### __assert_contains

In [ ]:
inl rec __assert_contains forall t u. (trace : testing_trace) (b : t) (a : u) : () =
    __expect
        fun a b =>
            run_target function
                | Rust => fun () =>
                    inl found : bool =
                        real
                            typecase u with
                            | string =>
                                typecase t with
                                | char => $'!a.contains(!b as char)' : bool
                                | _ => $'!a.contains(&*!b)' : bool
                            | array_base _ => $'!a.borrow().contains(&!b)' : bool
                            | _ => $'!a.iter().any(|x| *x == !b)' : bool
                    found
                | Zig => fun () =>
                    inl found : bool =
                        real
                            typecase u with
                            | string =>
                                typecase t with
                                | char => $'(std.mem.indexOfScalar(u8, !a, !b) \!= null)' : bool
                                | _ => $'(std.mem.indexOf(u8, !a, !b) \!= null)' : bool
                            | _ =>
                                typecase t with
                                | string => $'(for (!a) |x| { if (std.mem.eql(u8, x, !b)) break true; } else false)' : bool
                                | _ => $'(for (!a) |x| { if (x == !b) break true; } else false)' : bool
                    found
                | _ => fun () =>
                    a
                    |> $'List.ofSeq'
                    |> fun x => x : listm'.list' t
                    |> $'List.tryFind' ((=) b)
                    |> optionm'.unbox
                    |> fun (x : option t) => x <> None
        trace
        "__assert_contains"
        b
        a

inl _assert_contains b a =
    __assert_contains Console b a

In [5]:
///- --test

;[ "a"; "b"; "c" ]
|> _assert_contains "b"

__assert_contains { expected = b }

In [6]:
///- --test

"abcd"
|> _assert_contains 'b'

__assert_contains { expected = b }

### _throws

In [ ]:
inl _throws (fn : () -> ()) : option exn =
    inl none = None : option exn
    inl some (s : exn) = Some s
    if is_rust ()
    then
        catch_unwind_rust
            fun () =>
                fn ()
                none
            some
    else
        backend_switch {
            Fsharp = fun () =>
                $'try !fn (); !none with ex -> ex |> !some ' : option exn
            Python = fun () =>
                $'fn = !fn '
                $'none = !none '
                $'some = !some '
                $'try: fn(); x = none '
                $'except Exception as ex: x = some(ex)'
                $'x' : option exn
            TypeScript = fun () =>
                $'(() => { try { !fn(); return !none; } catch (ex) { return !some(ex); } })()' : option exn
            Zig = fun () =>
                fn ()
                none
        }

### print_and_return

In [7]:
inl rec print_and_return x =
    inl name = reflection.nameof { print_and_return }
    name ++#? { x }
    |> console.write_line
    x